In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample_submission = pd.read_csv('data/sample_submission.csv')

# Display the first few rows of each DataFrame
display(train.head())
display(test.head())
display(sample_submission.head())

,created_at,default_profile,default_profile_image,description,favourites_count,followers_count,friends_count,geo_enabled,id,lang,location,profile_background_image_url,profile_image_url,screen_name,statuses_count,verified,average_tweets_per_day,account_age_days,target
0,2012-01-15 23:40:09,True,False,Cosplayer/Fitness lover. Come to me https://t....,74,7,0,False,465096524,en,unknown,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/9666745212...,reml5477,20,False,0.006,3138,1
1,2016-10-04 00:44:39,False,False,pobody’s nerfect,50443,164,590,True,783105517673648132,cy,she/her,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1281752126...,kinlibra,6469,False,4.572,1415,0
2,2009-05-23 04:04:13,False,False,gracias por participar 🏅,9394,208,189,False,41970759,es,La diaspora,http://abs.twimg.com/images/themes/theme17/bg.gif,http://pbs.twimg.com/profile_images/1233811596...,_delaualau,30296,False,7.378,4106,0
3,2009-05-17 04:31:31,False,False,Stand Up Comedian/Actor from North Philadelphi...,46,66180,1090,True,40607946,en,"Calabasas, CA",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1184851104...,SpankHorton,164957,False,40.116,4112,0
4,2009-02-16 13:11:21,True,False,Assignment Editor at NBC10 and President of Ja...,1223,487,867,True,20983433,en,"Jenkintown, PA",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/5234863934...,javelinjt,1752,False,0.417,4201,0


,index,created_at,default_profile,default_profile_image,description,favourites_count,followers_count,friends_count,geo_enabled,id,lang,location,profile_background_image_url,profile_image_url,screen_name,statuses_count,verified,average_tweets_per_day,account_age_days
0,0,2007-03-27 22:56:04,False,False,Dad. #ADHD @fasternormal Pod host. Author/keyn...,22290,163813,1746,True,2569261,en,"New York, NY",http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1186468889...,petershankman,42013,True,8.586,4893
1,1,2013-03-30 08:47:01,True,False,NaN,685,44,637,False,1316038452,NaN,unknown,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/9637876928...,Mihail_HD,111,False,0.041,2699
2,2,2009-04-25 15:24:08,False,False,#UGAMOM #GODAWGS,12257,276,194,True,35233977,de,"Augusta, GA",http://abs.twimg.com/images/themes/theme3/bg.gif,http://pbs.twimg.com/profile_images/1246470377...,TinkerMazell,18142,False,4.390,4133
3,3,2010-08-06 22:21:56,True,False,"golfer, traveler, car guy, PCA member, owns 19...",30806,1124,4999,True,175542625,en,New York,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/1391357162...,joecangolfICC,20624,False,5.627,3665
4,4,2015-03-02 12:29:39,True,False,Do you like it gently? Come in! 💚💚💚 https://t....,224,22,265,False,3056068536,en,unknown,http://abs.twimg.com/images/themes/theme1/bg.png,http://pbs.twimg.com/profile_images/9680856906...,TiesiogDovydas,64,False,0.032,1996


,index,target
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


## Simple Model for Bot Account Prediction

We will train a basic logistic regression model using the features in the training set to predict which accounts are bots. The predictions will be formatted according to the sample submission.

In [11]:
# Prepare features and target
X = train.drop(columns=['target'])
y = train['target']

# For simplicity, drop non-numeric columns (basic baseline)
X = X.select_dtypes(include=['number'])
test_X = test[X.columns]

# Split for quick validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Validation accuracy
val_preds = model.predict(X_val)
print('Validation Accuracy:', accuracy_score(y_val, val_preds))

# Predict on test set
preds = model.predict(test_X)

# Prepare submission
submission = sample_submission.copy()
submission['target'] = preds
submission.to_csv('submission.csv', index=False)
submission.head()

Validation Accuracy: 0.6772224341854254


,index,target
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


## Improved Modeling Pipeline

We will:
- Encode categorical features
- Impute missing values
- Try Random Forest and XGBoost models
- Use cross-validation and hyperparameter tuning
- Analyze feature importance

In [6]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

In [7]:
# Define your specific feature types
numeric_features = [
    'favourites_count', 'followers_count', 'friends_count',
    'average_tweets_per_day', 'account_age_days'
]

binary_features = ['default_profile', 'default_profile_image', 'geo_enabled', 'verified']

categorical_features = ['lang', 'location']

keep_columns = numeric_features + binary_features + categorical_features + ['target']

# Filter your data to keep only the specified columns
train = train[keep_columns]
test = test[keep_columns[:-1]] if 'target' not in test.columns else test[keep_columns]

# Check and print data types before preprocessing
print("Data types before preprocessing:")
print(train.dtypes)
print("\nUnique values in binary features:")
for col in binary_features:
    if col in train.columns:
        print(f"{col}: {train[col].unique()}")

# Convert binary features to integers (0/1) to handle boolean dtype
for col in binary_features:
    if col in train.columns:
        train[col] = train[col].astype('int64')
    if col in test.columns:
        test[col] = test[col].astype('int64')

# Separate features and target
y = train['target']
X = train.drop(columns=['target'])

# Create preprocessing pipelines for each feature type
numeric_transformer = SimpleImputer(strategy='mean')

# Binary features: Convert to int and use mean imputation (will be 0 or 1)
binary_transformer = SimpleImputer(strategy='most_frequent')

# Categorical features: Imputation + One-Hot Encoding
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Verify that we have the right columns for each transformer
print("Verifying feature assignments:")
print(f"Numeric features: {[col for col in numeric_features if col in X.columns]}")
print(f"Binary features: {[col for col in binary_features if col in X.columns]}")
print(f"Categorical features: {[col for col in categorical_features if col in X.columns]}")

# Only include features that actually exist in the dataset
available_numeric = [col for col in numeric_features if col in X.columns]
available_binary = [col for col in binary_features if col in X.columns]
available_categorical = [col for col in categorical_features if col in X.columns]

# Create the preprocessor with your specific feature groups
transformers = []
if available_numeric:
    transformers.append(('num', numeric_transformer, available_numeric))
if available_binary:
    transformers.append(('bin', binary_transformer, available_binary))
if available_categorical:
    transformers.append(('cat', categorical_transformer, available_categorical))

preprocessor = ColumnTransformer(transformers, remainder='drop')  # Drop any columns not specified

# Fit preprocessor on train and transform both train and test
X_processed = preprocessor.fit_transform(X)

# For test set, make sure it has the same columns (excluding target)
if 'target' in test.columns:
    test_features = test.drop(columns=['target'])
else:
    test_features = test

test_processed = preprocessor.transform(test_features)

# Split for validation
X_train, X_val, y_train, y_val = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Random Forest
print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print('Random Forest training complete.')
print('Predicting on validation set with Random Forest...')
rf_val_preds = rf.predict(X_val)
print('Random Forest Validation Accuracy:', accuracy_score(y_val, rf_val_preds))
print('Random Forest Classification Report:')
print(classification_report(y_val, rf_val_preds))

# XGBoost
print('Training XGBoost...')
xgb = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)
print('XGBoost training complete.')
print('Predicting on validation set with XGBoost...')
xgb_val_preds = xgb.predict(X_val)
print('XGBoost Validation Accuracy:', accuracy_score(y_val, xgb_val_preds))
print('XGBoost Classification Report:')
print(classification_report(y_val, xgb_val_preds))

# Cross-validation (Random Forest)
print('Performing 5-fold cross-validation with Random Forest...')
rf_cv_scores = cross_val_score(rf, X_processed, y, cv=5)
print('Random Forest CV Accuracy:', np.mean(rf_cv_scores))

# Feature importance (Random Forest) with proper feature names
if hasattr(rf, 'feature_importances_'):
    print('Calculating feature importances for Random Forest...')
    importances = rf.feature_importances_
    
    # Get feature names after preprocessing
    feature_names = []
    
    # Add numeric feature names (only those that exist)
    feature_names.extend(available_numeric)
    
    # Add binary feature names (only those that exist)
    feature_names.extend(available_binary)
    
    # Add categorical feature names (after one-hot encoding, only if any exist)
    if available_categorical:
        cat_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(available_categorical)
        feature_names.extend(cat_feature_names)
    
    # Create feature importance dataframe
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print('Top 10 Most Important Features:')
    print(feature_importance_df.head(10))

# Optional: Print the shape and feature count after preprocessing
print(f'\nProcessed training data shape: {X_processed.shape}')
print(f'Number of features after preprocessing: {X_processed.shape[1]}')
print(f'Feature breakdown:')
print(f'  - Numeric features: {len(available_numeric)}')
print(f'  - Binary features: {len(available_binary)}')
print(f'  - Categorical features (before encoding): {len(available_categorical)}')
if available_categorical:
    total_cat_features = len(preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(available_categorical))
    print(f'  - Categorical features (after one-hot encoding): {total_cat_features}')

Data types before preprocessing:
favourites_count            int64
followers_count             int64
friends_count               int64
average_tweets_per_day    float64
account_age_days            int64
default_profile              bool
default_profile_image        bool
geo_enabled                  bool
verified                     bool
lang                       object
location                   object
target                      int64
dtype: object

Unique values in binary features:
default_profile: [ True False]
default_profile_image: [False  True]
geo_enabled: [False  True]
verified: [False  True]
Verifying feature assignments:
Numeric features: ['favourites_count', 'followers_count', 'friends_count', 'average_tweets_per_day', 'account_age_days']
Binary features: ['default_profile', 'default_profile_image', 'geo_enabled', 'verified']
Categorical features: ['lang', 'location']
Training Random Forest...
Training Random Forest...
Random Forest training complete.
Predicting on validati

/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [17:02:12] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost training complete.
Predicting on validation set with XGBoost...
XGBoost Validation Accuracy: 0.884586035864174
XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.93      0.92      3550
           1       0.85      0.79      0.81      1692

    accuracy                           0.88      5242
   macro avg       0.87      0.86      0.87      5242
weighted avg       0.88      0.88      0.88      5242

Performing 5-fold cross-validation with Random Forest...
Random Forest CV Accuracy: 0.8693430448636681
Calculating feature importances for Random Forest...
Top 10 Most Important Features:
                     feature  importance
1            followers_count    0.148786
0           favourites_count    0.143012
2              friends_count    0.134249
3     average_tweets_per_day    0.115021
4           account_age_days    0.093552
8                   verified    0.033302
7                geo_enabled    0.030469
5      

In [8]:
# =============================================================================
# MODEL SELECTION AND FINAL PREDICTIONS
# =============================================================================

# Choose the best model based on validation performance
# You can modify this logic based on your validation results
print('\n' + '='*50)
print('FINAL MODEL SELECTION AND SUBMISSION')
print('='*50)

# Compare model performances (you can add more sophisticated model selection logic)
rf_val_acc = accuracy_score(y_val, rf_val_preds)
xgb_val_acc = accuracy_score(y_val, xgb_val_preds)

print(f'Random Forest Validation Accuracy: {rf_val_acc:.4f}')
print(f'XGBoost Validation Accuracy: {xgb_val_acc:.4f}')

# Select the best model
if rf_val_acc >= xgb_val_acc:
    best_model = rf
    best_model_name = 'Random Forest'
    print(f'\nSelected model: {best_model_name}')
else:
    best_model = xgb
    best_model_name = 'XGBoost'
    print(f'\nSelected model: {best_model_name}')

# Train the final model on all training data (optional - for better performance)
print(f'Training final {best_model_name} model on all training data...')
final_model = best_model.__class__(**best_model.get_params())
final_model.fit(X_processed, y)
print('Final model training complete.')

# =============================================================================
# PREDICTIONS ON TEST SET AND SUBMISSION
# =============================================================================

print('\nMaking predictions on test set...')

# Predict probabilities and binary predictions
test_preds_proba = final_model.predict_proba(test_processed)[:, 1]
test_preds = (test_preds_proba >= 0.5).astype(int)

print(f'Test predictions shape: {test_preds.shape}')
print(f'Test predictions distribution: {np.bincount(test_preds)}')
print(f'Test prediction probabilities - Min: {test_preds_proba.min():.4f}, Max: {test_preds_proba.max():.4f}, Mean: {test_preds_proba.mean():.4f}')

# Create submission file
# Note: Make sure you have 'sample_submission' loaded or create the submission format
try:
    submission = sample_submission.copy()
    submission['target'] = test_preds
    print('\nUsing provided sample_submission format.')
except NameError:
    # If sample_submission is not available, create a basic submission format
    # Assuming test set has an ID column or we use index
    print('\nCreating submission format (assuming test index as ID)...')
    submission = pd.DataFrame({
        'id': range(len(test_preds)),  # Adjust this based on your test set structure
        'target': test_preds
    })

# Save submission
submission.to_csv('submission.csv', index=False)
print('Submission saved to submission.csv')
print('\nSubmission preview:')
print(submission.head())

# Optional: Also save predictions with probabilities for analysis
detailed_submission = submission.copy()
detailed_submission['target_proba'] = test_preds_proba
detailed_submission.to_csv('submission_with_probabilities.csv', index=False)
print('\nDetailed submission with probabilities saved to submission_with_probabilities.csv')

print(f'\nFinal Summary:')
print(f'- Best model: {best_model_name}')
print(f'- Validation accuracy: {max(rf_val_acc, xgb_val_acc):.4f}')
print(f'- Test predictions: {len(test_preds)} samples')
print(f'- Positive predictions: {sum(test_preds)} ({sum(test_preds)/len(test_preds)*100:.1f}%)')
print('- Submission file: submission.csv')


FINAL MODEL SELECTION AND SUBMISSION
Random Forest Validation Accuracy: 0.8735
XGBoost Validation Accuracy: 0.8846

Selected model: XGBoost
Training final XGBoost model on all training data...


/Users/winston/Documents/School/Y3S1/BT4012/Kaggle/Competition 1/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [17:08:13] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Final model training complete.

Making predictions on test set...
Test predictions shape: (11232,)
Test predictions distribution: [7945 3287]
Test prediction probabilities - Min: 0.0000, Max: 0.9992, Mean: 0.3250

Using provided sample_submission format.
Submission saved to submission.csv

Submission preview:
   index  target
0      0       0
1      1       1
2      2       0
3      3       0
4      4       1

Detailed submission with probabilities saved to submission_with_probabilities.csv

Final Summary:
- Best model: XGBoost
- Validation accuracy: 0.8846
- Test predictions: 11232 samples
- Positive predictions: 3287 (29.3%)
- Submission file: submission.csv


In [2]:
# Hyperparameter tuning for XGBoost (optional, can take time)
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200]
}
gs = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
                  param_grid, cv=3, scoring='accuracy', verbose=1, n_jobs=-1)
gs.fit(X_train, y_train)
print('Best XGBoost Params:', gs.best_params_)
print('Best CV Score:', gs.best_score_)

# Predict with best estimator
gs_val_preds = gs.predict(X_val)
print('Tuned XGBoost Validation Accuracy:', accuracy_score(y_val, gs_val_preds))

NameError: name 'X_train' is not defined

In [18]:
import joblib

# Save the best XGBoost model
print('Saving the best XGBoost model to best_xgb_model.joblib...')
joblib.dump(gs.best_estimator_, 'best_xgb_model.joblib')
print('Model saved.')

# Predict on the test set using the best estimator
print('Predicting on test set with the best XGBoost model...')
test_preds = gs.best_estimator_.predict(test_processed)

# Prepare and save the submission file
submission = sample_submission.copy()
submission['target'] = test_preds
submission.to_csv('submission.csv', index=False)
print('Submission file saved as submission.csv')
submission.head()

Saving the best XGBoost model to best_xgb_model.joblib...
Model saved.
Predicting on test set with the best XGBoost model...
Submission file saved as submission.csv


,index,target
0,0,0
1,1,1
2,2,0
3,3,0
4,4,1
